<a href="https://colab.research.google.com/github/saeedbhai0632-web/Math-to-Magic/blob/main/FashionMNIST_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Learn The Basics**


Pytorch gives alot of datasets like torchvision torchaudio torchtext, for now we will be using torchvision


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

-TorchVision contains alot of vision based objects called 'Datasets' the object we will use today is FashionMNIST.
-Each dataset has two arguments target_transform and transform



In [2]:
# Download training data from open datasets.
training_data=datasets.FashionMNIST(
    root="data",
    train="true",
    download="true",
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

creating dataloader:

In [3]:
batch_size = 64

# Create data loaders.
train_dataloader=DataLoader(training_data,batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)#variable and parameter can have same name

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


using GPU if avaiable, if not, use CPU

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


creating the model:

In [5]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


loss calculator and optimizer(changes the weights and biases)

In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

training loop:

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

testing loop


In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

output


In [9]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.293984  [   64/60000]
loss: 2.294073  [ 6464/60000]
loss: 2.267820  [12864/60000]
loss: 2.265551  [19264/60000]
loss: 2.262325  [25664/60000]
loss: 2.213571  [32064/60000]
loss: 2.237316  [38464/60000]
loss: 2.199696  [44864/60000]
loss: 2.185701  [51264/60000]
loss: 2.176162  [57664/60000]
Epoch 2
-------------------------------
loss: 2.167812  [   64/60000]
loss: 2.171607  [ 6464/60000]
loss: 2.110979  [12864/60000]
loss: 2.128308  [19264/60000]
loss: 2.098091  [25664/60000]
loss: 2.016533  [32064/60000]
loss: 2.060918  [38464/60000]
loss: 1.987339  [44864/60000]
loss: 1.978441  [51264/60000]
loss: 1.920213  [57664/60000]
Epoch 3
-------------------------------
loss: 1.945173  [   64/60000]
loss: 1.928573  [ 6464/60000]
loss: 1.811667  [12864/60000]
loss: 1.847400  [19264/60000]
loss: 1.751088  [25664/60000]
loss: 1.682594  [32064/60000]
loss: 1.716070  [38464/60000]
loss: 1.619935  [44864/60000]
loss: 1.630232  [51264/60000]
loss: 1.52